# State-wise cane dues

I put the official state-wise cane dues tables into one long table here. Each row is a state, a sugar season and the date the figures were taken (the snapshot). All three snapshots are from mid or late July, right after crushing ends, so the current season row in each one shows how much was still unpaid at roughly the same point in the season.

The 2024 reply is a scanned PDF, so I typed its two tables into `ls_dues_2024_typed.csv` and checked every row and column against the printed totals.

In [1]:
import pandas as pd
from lxml import html

RAW = "data/raw/"

# state names are spelt a bit differently across releases, so I map them to one spelling
FIX = {"Uttrakhand": "Uttarakhand", "Total": "All India"}


def num(x):
    x = x.replace(",", "").strip()
    return float(x) if x else None


def table_rows(path):
    # the annexure sits inside a wrapper table on PIB pages, so I take the innermost table that names UP
    doc = html.parse(path)
    for t in doc.iter("table"):
        if t.find(".//table") is None and "Uttar" in t.text_content():
            return [[" ".join(td.text_content().split()) for td in tr.iter("td")] for tr in t.iter("tr")]
    raise ValueError("no state table in " + path)


def tidy(df):
    df["state"] = df["state"].str.strip().replace(FIX)
    return df

July 2019 only gives arrears, no payable or paid amounts.

In [2]:
# July 2019: only arrears, for the current and last season
rows = table_rows(RAW + "pib_cane_dues_2019-07-19.html")
rec = []
for r in rows:
    if len(r) == 5 and r[2].replace(",", "").strip().isdigit():
        rec.append({"state": r[1], "season": "2018-19", "arrear": num(r[2])})
        rec.append({"state": r[1], "season": "2017-18", "arrear": num(r[3])})
d19 = tidy(pd.DataFrame(rec))
d19["snapshot"] = "2019-07-16"
d19["source"] = "PIB 19 Jul 2019 (RS reply)"

In [3]:
# July 2023: payable, paid and arrear for five seasons
rows = table_rows(RAW + "pib_cane_dues_2023-07-21.html")
seasons = [c.replace("SS", "").strip() for c in rows[2][1:]]
rec = []
for r in rows[4:]:
    for k, s in enumerate(seasons):
        rec.append({"state": r[0], "season": s, "payable": num(r[1 + 3 * k]),
                    "paid": num(r[2 + 3 * k]), "arrear": num(r[3 + 3 * k])})
d23 = tidy(pd.DataFrame(rec))
d23["snapshot"] = "2023-07-17"
d23["source"] = "PIB 21 Jul 2023 (RS USQ 227)"

In [4]:
# July 2024: typed by hand from the scanned LS reply, annexure II has full rows,
# annexure I adds arrears for the older seasons that II does not cover
t24 = pd.read_csv(RAW + "ls_dues_2024_typed.csv")
full = t24[t24["annexure"] == "II"]
older = t24[(t24["annexure"] == "I") & ~t24["season"].isin(full["season"])]
d24 = tidy(pd.concat([full, older]).drop(columns="annexure"))
d24["snapshot"] = "2024-07-31"
d24["source"] = "LS USQ 2624, 7 Aug 2024"

In [5]:
dues = pd.concat([d19, d23, d24], ignore_index=True)
dues = dues[["snapshot", "source", "state", "season", "payable", "paid", "arrear"]]
dues["paid_share"] = (dues["paid"] / dues["payable"] * 100).round(2)
dues = dues.sort_values(["snapshot", "season", "state"]).reset_index(drop=True)

Every season in every snapshot should add up to the printed All India line, so the gap column should be all zeros.

In [6]:
# check that the states add up to the printed All India line in each snapshot and season
check = (dues[dues["state"] != "All India"].groupby(["snapshot", "season"])["arrear"].sum()
         .to_frame("states").join(dues[dues["state"] == "All India"].set_index(["snapshot", "season"])["arrear"]))
check["gap"] = check["states"] - check["arrear"]
check

states   arrear  gap
snapshot   season                                  
2019-07-16 2017-18              285.0    285.0  0.0
           2018-19            15565.0  15565.0  0.0
2023-07-17 2018-19              277.0    277.0  0.0
           2019-20               77.0     77.0  0.0
           2020-21               73.0     73.0  0.0
           2021-22              115.0    115.0  0.0
           2022-23             9499.0   9499.0  0.0
2024-07-31 2018-19 & earlier    526.0    526.0  0.0
           2019-20               75.0     75.0  0.0
           2020-21               60.0     60.0  0.0
           2021-22               97.0     97.0  0.0
           2022-23              238.0    238.0  0.0
           2023-24             5188.0   5188.0  0.0

In [7]:
dues.to_csv("data/clean/state_dues.csv", index=False)
dues[dues["state"].isin(["Uttar Pradesh", "All India"])]

,snapshot,source,state,season,payable,paid,arrear,paid_share
0,2019-07-16,PIB 19 Jul 2019 (RS reply),All India,2017-18,NaN,NaN,285.0,NaN
15,2019-07-16,PIB 19 Jul 2019 (RS reply),Uttar Pradesh,2017-18,NaN,NaN,46.0,NaN
17,2019-07-16,PIB 19 Jul 2019 (RS reply),All India,2018-19,NaN,NaN,15565.0,NaN
32,2019-07-16,PIB 19 Jul 2019 (RS reply),Uttar Pradesh,2018-19,NaN,NaN,9746.0,NaN
34,2023-07-17,PIB 21 Jul 2023 (RS USQ 227),All India,2018-19,86723.0,86446.0,277.0,99.68
47,2023-07-17,PIB 21 Jul 2023 (RS USQ 227),Uttar Pradesh,2018-19,33048.0,33048.0,0.0,100.00
49,2023-07-17,PIB 21 Jul 2023 (RS USQ 227),All India,2019-20,75931.0,75854.0,77.0,99.90
62,2023-07-17,PIB 21 Jul 2023 (RS USQ 227),Uttar Pradesh,2019-20,35898.0,35898.0,0.0,100.00
64,2023-07-17,PIB 21 Jul 2023 (RS USQ 227),All India,2020-21,93075.0,93002.0,73.0,99.92
77,2023-07-17,PIB 21 Jul 2023 (RS USQ 227),Uttar Pradesh,2020-21,33014.0,33010.0,4.0,99.99
